# Helm 02: values are the context

`values.yaml` is the chart's public API. Environments layer files with `-f` (last wins, key by key) and `--set` for one-offs. `values.schema.json` is the only type check Helm has, and only when the chart ships one.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers/examples/02-helm && export HOME=/tmp && helm dependency update web >/dev/null 2>&1; helm template web web -n web --skip-tests -f values-prod.yaml --show-only templates/deployment.yaml | yq '.spec.replicas, .spec.template.spec.containers[0].resources'


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
helm template web web -n web --skip-tests -f values-prod.yaml --set replicaCount=5 --show-only templates/deployment.yaml | yq '.spec.replicas'


Now break the type: the schema rejects it before any template renders.


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
(helm template web web -n web --skip-tests -f values-prod.yaml --set replicaCount=three 2>&1 | head -4) || true


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
cat web/values.schema.json | jq '.properties.replicaCount, .properties.image.properties.tag'


`--debug` shows the computed values and the rendered text together, which is where template whitespace bugs are found.


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
helm template web web -n web --skip-tests -f values-dev.yaml --debug 2>&1 | sed -n '/COMPUTED VALUES/,/^---/p' | head -25


Try it: mark a field required in values.schema.json, render without it, and read the error.
